The per-patch metadetect catalogs are referred to as `object_shear_patch`dataset.
For overlaying detections on images, this is a good dataset to start with.
For more survey-level diagnostic plots, `object_shear_all`, which has several `object_shear_patch` concatenated would be convenient.

In [ ]:
# Standard imports that we will need often

import numpy as np
import matplotlib.pyplot as plt

Load the butler and get the metadetection config, for later use

In [ ]:
from lsst.daf.butler import Butler

butler = Butler(
    "/sdf/data/rubin/repo/dp2_prep",
    collections=["LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3"],
    skymap='lsst_cells_v2',
    instrument='LSSTCam',
)

metadetect_config = butler.get("metadetectionShear_config")

What bands were combined to detect objects and measure shear?

In [ ]:
metadetect_config.metadetect.shear_bands

What bands do we have photometry on?

In [ ]:
metadetect_config.photometry_bands

Let us pick a (per-patch) metadetect catalog and look at it.
Specify a data ID.
We will see later how to know what the valid data IDs are.

In [ ]:
dataId = {
    "tract": 9813,
    "patch": 42,
}

In [ ]:
object_shear_patch = butler.get("object_shear_patch", dataId=dataId)

I don't recommend converting pyarrow table to pandas DataFrame, except may be to get an overview.

In [ ]:
object_shear_patch.to_pandas()

If you need to look just at the columns, the schema (same for all catalogs) has the column names and a brief description.

In [ ]:
# Uncomment to see the schema
# object_shear_patch.schema

For this particular patch, let us look at a single band image.
For this, we will use the `afwDisplay` package, which is the standard way of looking at the single-band images

In [ ]:
multiple_cell_coadd = butler.get("deep_coadd_cell_predetection", band="i", **dataId)
stitched_coadd = multiple_cell_coadd.stitch()
stitched_coadd.set_cell_edges()  # So we can see the cells
exposure = stitched_coadd.asExposure()

In [ ]:
%matplotlib widget
# The widget backend can be finicky, you might need to turn it off.
# Try inline or notebook instead.
from lsst.afw.display import Display
afwDisplay_backend = "matplotlib"  # Use "firefly" for another interactive backend
disp = Display(frame=1, backend=afwDisplay_backend)
disp.scale("linear", "zscale")
disp.image(exposure)
disp.show()

Similarly, you can visualize the noise image as well

In [ ]:
noise_image = stitched_coadd.asMaskedImage(noise_index=0)

disp = Display(frame=2, backend=afwDisplay_backend)
disp.scale("linear", "zscale")
disp.image(noise_image)
disp.show()

Visualizing the PSFs at the centers of cells is literally a 1-line code.
This particular bit does not have a long-term guarantee, but it works for DP2.

In [ ]:
psf_mosaic = multiple_cell_coadd.explode().psf_image

In [ ]:
disp = Display(frame=3, backend=afwDisplay_backend)
disp.scale("linear", "zscale")
disp.image(psf_mosaic)
disp.show()

or simply as

In [ ]:
plt.imshow(psf_mosaic.array, interpolation=None, cmap="viridis")
plt.colorbar()

To visualize the tri-color image that is used to detect objects for metadetect, we have some more work to do.
To convert them into a 3-channel RGB image, let us use the `PrettyPictureTask`.
If the color gamut is not to your taste, you will need to adjust the configuration below.

In [ ]:
from lsst.pipe.tasks.prettyPictureMaker import ChannelRGBConfig, PrettyPictureTask

pretty_pic_config = PrettyPictureTask.ConfigClass()
pretty_pic_config.load("../../config/prettyPicture.py")

# These are hard-coded for riz for now
pretty_pic_config.channelConfig["r"] = ChannelRGBConfig(r=0, g=0, b=1)
pretty_pic_config.channelConfig["i"] = ChannelRGBConfig(r=0, g=1, b=0)
pretty_pic_config.channelConfig["z"] = ChannelRGBConfig(r=1, g=0, b=0)

In [ ]:
def get_singles(dataId, band):
    """A utility function to get image given a band.
    """
    exp = butler.get("deep_coadd_cell_predetection", band=band, **dataId)
    stitched_coadd = exp.stitch()
    stitched_coadd.set_cell_edges()
    image = stitched_coadd.asExposure()
    return image
        
def create_rgba(dataId, pretty_pic_config=pretty_pic_config):
    task = PrettyPictureTask(config=pretty_pic_config)
    pretty_image = task.run(
        {band: get_singles(dataId, band) for band in metadetect_config.metadetect.shear_bands}
    )
    pretty_image_RGB = pretty_image.outputRGB

    # There is a good chance that the width and height are misplaced.
    width, height, _ = pretty_image_RGB.shape
    
    image = np.empty((width, height), dtype=np.uint32)
    view = image.view(dtype=np.uint8).reshape((width, height, 4))
    view[:, :, 0] = pretty_image_RGB[:, :, 0]
    view[:, :, 1] = pretty_image_RGB[:, :, 1]
    view[:, :, 2] = pretty_image_RGB[:, :, 2]
    view[:, :, 3] = 255

    return image

In [ ]:
pretty_image_RGB = create_rgba(dataId)

In [ ]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import BoxZoomTool, HoverTool
output_notebook()

tools = "undo,redo,save,pan,zoom_in,zoom_out,wheel_zoom,"
p = figure(width=400, height=400,
           title=None,
           tools=tools,
           active_inspect=[],  # hover is nice to have, but not by default
          )
p.add_tools(BoxZoomTool(match_aspect=True))  # you never want to change the aspect ratio
hover = HoverTool()
hover.tooltips = [
    ("(x,y)", "($x, $y)"),
]
p.add_tools(hover)

image = pretty_image_RGB
p.image_rgba(
    image=[image],
    x=stitched_coadd.bbox.getMinX(),
    y=stitched_coadd.bbox.getMinY(),
    dw=3300,
    dh=3300,
)

show(p)

Let us overlay the no-shear catalog on this image. Remember to conver the pyarrow chunked arrays into regular NumPy arrays.

In [ ]:
cuts = object_shear_patch["metaStep"].to_numpy() == "ns"
x = object_shear_patch["x"].to_numpy()[cuts]
y = object_shear_patch["y"].to_numpy()[cuts]

p.scatter(
    x, y,
    line_color="cyan",
)

show(p)